# L01 · 확률·미분·PyTorch 생존 키트

## Goal

- log-prob·entropy·KL을 계산한다
- expectation gradient를 설명한다
- detach 뒤 gradient 흐름을 확인한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L01:toy:42").hexdigest()
print(f"lesson=L01 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L01 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:30408c6216764a44279ea7141e1fd584ff1b221be0556e06c7bbed32db722e95 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: 전체 지도 → **확률·미분** → policy gradient

$$H(p)=-\sum_i p_i\log p_i,\qquad D_{KL}(p\|q)=\sum_i p_i\log\frac{p_i}{q_i}$$

log-prob는 곱을 합으로 바꾸고 선택한 action의 민감도를 표현합니다. entropy는 분포의 퍼짐, KL은 방향이 있는 두 분포의 차이입니다. `detach`는 값을 유지하면서 그 경로의 gradient 소유권을 끊습니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** entropy를 키우고 forward KL을 줄이면 가장 큰 logit의 gradient 부호는 어떻게 될까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>두 항이 경쟁하므로 직감만으로 확정하지 말고 autograd 값을 읽어야 합니다. 이 입력에서는 두 번째 logit gradient가 음수입니다.</details>

In [2]:
from rl_study.math import categorical_entropy, categorical_kl
logits = torch.tensor([[0.0, 1.0, -1.0]], requires_grad=True)
other = torch.tensor([[0.4, 0.2, -0.3]])
entropy = categorical_entropy(logits)
forward_kl = categorical_kl(logits, other)
objective = entropy.mean() - forward_kl.mean()
objective.backward()
print({"entropy": round(float(entropy.detach()), 4),
       "kl_p_q": round(float(forward_kl.detach()), 4),
       "gradient": [round(x, 4) for x in logits.grad[0].tolist()]})

{'entropy': 0.8324, 'kl_p_q': 0.2032, 'gradient': [0.3295, -0.5678, 0.2383]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 확률을 직접 clamp하기보다 `log_softmax` 기반 package 함수를 쓰면 정규화와 수치 안정성을 함께 얻습니다. reverse KL은 다른 mode-seeking 성질을 가지므로 같은 값으로 취급할 수 없습니다.

**흔한 함정:** `float(tensor)`로 gradient tensor를 출력하면 경고가 납니다. 관찰용 값은 `detach()`한 뒤 scalar로 바꾸고, 학습 loss에는 detach하지 않습니다. 회귀 test: `test_probability_matches_torch`, `test_reinforce_sign`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert float(forward_kl.detach()) >= 0.0 and torch.isfinite(logits.grad).all()
print("checks=passed")

checks=passed


**회상 문제:** `D_KL(p||q)`와 `D_KL(q||p)`를 바꾸면 왜 같은 regularizer가 아닌가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** entropy와 KL은 유한했고 세 logit gradient의 합은 거의 0입니다. softmax가 공통 logit 이동에 불변이라는 사실과 맞습니다.
- 실제 확인: `test_probability_matches_torch`, `test_reinforce_sign`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L02에서 class 하나를 고르는 확률을 token sequence의 log-probability와 mask로 확장합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/math.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`